# Employee Attrition & Workforce Analytics
**Exploratory Data Analysis (EDA) using Python**

## 1. Project objective
A company is experiencing employee turnover. This notebook explores the **IBM HR Analytics Employee Attrition & Performance** dataset to understand which factors are associated with employees leaving, and converts the findings into HR recommendations.

**Key questions**
1. What is the overall attrition rate?
2. Which departments and job roles have higher attrition?
3. Does overtime relate to attrition?
4. Does income differ between employees who leave and stay?
5. How do job satisfaction and work-life balance relate to attrition?
6. Does distance from home or age matter?
7. What differences exist between employees who left and those who stayed?

**Dataset:** [IBM HR Analytics Attrition Dataset (Kaggle)](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset), file `data/WA_Fn-UseC_-HR-Employee-Attrition.csv`. This is a fictional dataset created by IBM data scientists, which matters for the limitations section at the end.

## 2. Setup

In [1]:
%pip install pandas numpy matplotlib seaborn scipy


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
PAL = {"No": "#4C78A8", "Yes": "#E4572E"}   # Stayed = blue, Left = orange-red
os.makedirs("figures", exist_ok=True)

def save(name):
    """Save the current figure for the report/slides, then show it."""
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

## 3. Data understanding

In [3]:
df = pd.read_csv("/Users/YourName/Downloads/Project02_Employee_Attrition/data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print("Rows, columns:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/YourName/Downloads/Project02_Employee_Attrition/data/WA_Fn-UseC_-HR-Employee-Attrition.csv'

In [ ]:
df.info()

In [ ]:
# Basic data dictionary (type, number of unique values, meaning)
desc = {
 "Age":"Employee age in years","Attrition":"Whether the employee left (Yes/No) - TARGET","BusinessTravel":"Travel frequency for work",
 "DailyRate":"Daily pay rate","Department":"Department","DistanceFromHome":"Distance from home to work (miles)",
 "Education":"1 Below College, 2 College, 3 Bachelor, 4 Master, 5 Doctor","EducationField":"Field of study",
 "EmployeeCount":"Always 1 (constant)","EmployeeNumber":"Employee ID","EnvironmentSatisfaction":"1 Low - 4 Very High",
 "Gender":"Gender","HourlyRate":"Hourly pay rate","JobInvolvement":"1 Low - 4 Very High","JobLevel":"Seniority level 1-5",
 "JobRole":"Job role","JobSatisfaction":"1 Low - 4 Very High","MaritalStatus":"Single / Married / Divorced",
 "MonthlyIncome":"Monthly income (USD)","MonthlyRate":"Monthly rate","NumCompaniesWorked":"Number of previous employers",
 "Over18":"Always Y (constant)","OverTime":"Works overtime (Yes/No)","PercentSalaryHike":"Last salary hike %",
 "PerformanceRating":"1 Low - 4 Outstanding","RelationshipSatisfaction":"1 Low - 4 Very High","StandardHours":"Always 80 (constant)",
 "StockOptionLevel":"Stock option level 0-3","TotalWorkingYears":"Total career years","TrainingTimesLastYear":"Trainings last year",
 "WorkLifeBalance":"1 Bad - 4 Best","YearsAtCompany":"Years at this company","YearsInCurrentRole":"Years in current role",
 "YearsSinceLastPromotion":"Years since last promotion","YearsWithCurrManager":"Years with current manager"}
dd = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_unique": df.nunique(),
                   "example": df.iloc[0], "meaning": pd.Series(desc)})
dd

In [ ]:
df.describe().T.round(2)

## 4. Data cleaning
Cleaning decisions are documented below.

In [ ]:
# 4.1 Missing values and duplicates
print("Total missing values :", int(df.isna().sum().sum()))
print("Duplicate rows       :", int(df.duplicated().sum()))
print("Duplicate EmployeeNumber:", int(df["EmployeeNumber"].duplicated().sum()))

In [ ]:
# 4.2 Constant / identifier columns carry no information -> drop
const_cols = [c for c in df.columns if df[c].nunique() == 1]
print("Constant columns:", const_cols)
df = df.drop(columns=const_cols + ["EmployeeNumber"])

# 4.3 Numeric target for correlations & rate calculations
df["Attrition_flag"] = (df["Attrition"] == "Yes").astype(int)

# 4.4 Human-readable labels for the 1-4 coded survey columns (kept alongside the numeric version)
sat = {1: "Low", 2: "Medium", 3: "High", 4: "Very High"}
wlb = {1: "Bad", 2: "Good", 3: "Better", 4: "Best"}
df["JobSatisfaction_lbl"] = df["JobSatisfaction"].map(sat)
df["WorkLifeBalance_lbl"] = df["WorkLifeBalance"].map(wlb)

# 4.5 Age bands for segmentation
df["AgeGroup"] = pd.cut(df["Age"], bins=[17, 25, 35, 45, 55, 61], labels=["18-25", "26-35", "36-45", "46-55", "56+"])

# 4.6 Logical consistency checks
print("YearsAtCompany > TotalWorkingYears :", int((df.YearsAtCompany > df.TotalWorkingYears).sum()))
print("YearsInCurrentRole > YearsAtCompany:", int((df.YearsInCurrentRole > df.YearsAtCompany).sum()))
print("Age < 18                           :", int((df.Age < 18).sum()))
print("MonthlyIncome <= 0                 :", int((df.MonthlyIncome <= 0).sum()))
print("Shape after dropping 4 columns and adding 4 helper columns:", df.shape)

In [ ]:
# 4.7 Outlier detection with the IQR rule (investigated, not removed)
num_cols = ["Age", "MonthlyIncome", "DistanceFromHome", "YearsAtCompany", "TotalWorkingYears",
            "YearsSinceLastPromotion", "NumCompaniesWorked", "TrainingTimesLastYear"]
rows = []
for c in num_cols:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    n = int(((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum())
    rows.append((c, n, round(100 * n / len(df), 1), round(skew := df[c].skew(), 2)))
outl = pd.DataFrame(rows, columns=["column", "n_outliers", "pct", "skewness"]).set_index("column")
print(outl)

hi = df[df.MonthlyIncome > df.MonthlyIncome.quantile(.75) + 1.5 * stats.iqr(df.MonthlyIncome)]
print("\nHigh-income outliers by job role:\n", hi.JobRole.value_counts().to_string())
print("\nHigh-income outliers by job level:\n", hi.JobLevel.value_counts().sort_index().to_string())

**Cleaning decisions**
- **No missing values and no duplicate rows**, so nothing needed imputing or removing.
- **Dropped 4 columns:** `EmployeeCount`, `Over18`, `StandardHours` (identical for every employee) and `EmployeeNumber` (an ID).
- **Created** `Attrition_flag` (1/0), readable labels for satisfaction and work-life balance, and `AgeGroup`.
- **Outliers were kept.** The high-income outliers are almost all senior roles (Managers, Research Directors) at job levels 4-5, so they are genuine, not errors. Removing them would delete real senior employees. Skewed columns are analysed with medians and box plots.
- Logical checks (tenure vs career length, age >= 18, positive income) all passed.

## 5. Exploratory analysis
### 5.1 Overall attrition rate (Q1)

In [ ]:
overall = df.Attrition_flag.mean() * 100
print(df.Attrition.value_counts().to_string())
print(f"\nOverall attrition rate: {overall:.1f}%  ({df.Attrition_flag.sum()} of {len(df)} employees left)")

fig, ax = plt.subplots(figsize=(5.5, 4))
sns.countplot(data=df, x="Attrition", hue="Attrition", palette=PAL, order=["No", "Yes"], legend=False, ax=ax)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=11)
ax.set_title(f"Employees who left vs stayed (attrition = {overall:.1f}%)")
ax.set_xticks([0, 1], ["Stayed", "Left"]); ax.set_xlabel("")
save("01_count_attrition")

**What it shows:** 16.1% of employees (237 of 1470) left. The data is **imbalanced** (about 1 leaver for every 5.2 stayers), so we compare *rates* between groups rather than raw counts, otherwise big groups always look like they have the most leavers.

### 5.2 Univariate analysis of key numeric variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (c, col, t) in zip(axes, [("Age", "#4C78A8", "Age"), ("MonthlyIncome", "#59A14F", "Monthly income (USD)"),
                                  ("DistanceFromHome", "#B279A2", "Distance from home (miles)")]):
    sns.histplot(df[c], bins=25, kde=True, color=col, ax=ax)
    ax.axvline(df[c].median(), color="k", ls="--", lw=1, label=f"median {df[c].median():,.0f}")
    ax.set_title(f"Distribution of {t}"); ax.set_xlabel(t); ax.legend()
save("02_histograms")
print(df[["Age", "MonthlyIncome", "DistanceFromHome"]].agg(["mean", "median", "skew"]).round(2))

**What it shows:** Age is roughly bell-shaped (median 36). **MonthlyIncome is strongly right-skewed** (skewness 1.37; mean $6,503 vs median $4,919): most staff earn modestly and a small senior group earns much more, so the median is the fairer summary. DistanceFromHome is also right-skewed: most employees live close to work, with a long tail of long commutes.

### 5.3 Attrition by department, job role and overtime (Q2, Q3, Q7)

In [ ]:
def rate_by(col):
    return (df.groupby(col)["Attrition_flag"].agg(["mean", "sum", "count"])
              .assign(rate=lambda d: d["mean"] * 100).sort_values("rate", ascending=False))

dept = rate_by("Department"); role = rate_by("JobRole"); ot = rate_by("OverTime")
print(dept[["sum", "count", "rate"]].round(1), "\n")
print(role[["sum", "count", "rate"]].round(1), "\n")
print(ot[["sum", "count", "rate"]].round(1))

# Bar chart 1: department
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=dept.index, y=dept.rate, hue=dept.index, palette="Blues_r", legend=False, ax=ax)
ax.axhline(overall, color="red", ls="--", label=f"company average {overall:.1f}%")
for i, v in enumerate(dept.rate): ax.text(i, v + .3, f"{v:.1f}%", ha="center")
ax.set_ylabel("Attrition rate (%)"); ax.set_xlabel(""); ax.set_title("Attrition rate by department"); ax.legend()
save("03_bar_department")

# Bar chart 2: job role
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(y=role.index, x=role.rate, hue=role.index, palette="Reds_r", legend=False, ax=ax)
ax.axvline(overall, color="k", ls="--", label=f"company average {overall:.1f}%")
for i, v in enumerate(role.rate): ax.text(v + .3, i, f"{v:.1f}%", va="center")
ax.set_xlabel("Attrition rate (%)"); ax.set_ylabel(""); ax.set_title("Attrition rate by job role"); ax.legend(loc="lower right")
save("04_bar_jobrole")

# Bar chart 3: overtime
fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=ot.index, y=ot.rate, hue=ot.index, palette={"Yes": "#E4572E", "No": "#4C78A8"}, legend=False, ax=ax)
for i, v in enumerate(ot.rate): ax.text(i, v + .5, f"{v:.1f}%", ha="center")
ax.set_xlabel("Works overtime"); ax.set_ylabel("Attrition rate (%)"); ax.set_title("Attrition rate: overtime vs no overtime")
save("05_bar_overtime")

# Significance test (chi-square) - is overtime related to attrition?
chi_ot = stats.chi2_contingency(pd.crosstab(df.OverTime, df.Attrition))
chi_dept = stats.chi2_contingency(pd.crosstab(df.Department, df.Attrition))
print(f"\nChi-square p-values -> OverTime: {chi_ot[1]:.2e}, Department: {chi_dept[1]:.3f}")

**What it shows**
- **Overtime is the strongest single signal:** employees working overtime leave at **30.5%** versus **10.4%** without overtime, about **2.9x** higher (chi-square p = 8.2e-21, so this is very unlikely to be chance).
- **Departments:** Sales has the highest attrition at 20.6% and Research & Development the lowest at 13.8% (company average 16.1%). The department differences are statistically significant (chi-square p = 0.005) but modest in size compared with overtime.
- **Job roles:** Sales Representative (39.8%) and Laboratory Technician (23.9%) lose the largest share of staff, while Research Director is the most stable at 2.5%.

### 5.4 Income, age, distance and tenure: leavers vs stayers (Q4, Q6, Q8)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
for ax, (c, t) in zip(axes, [("MonthlyIncome", "Monthly income (USD)"), ("Age", "Age (years)"),
                             ("DistanceFromHome", "Distance from home (miles)"), ("YearsAtCompany", "Years at company")]):
    sns.boxplot(data=df, x="Attrition", y=c, hue="Attrition", palette=PAL, order=["No", "Yes"], legend=False, ax=ax)
    ax.set_xticks([0, 1], ["Stayed", "Left"]); ax.set_xlabel(""); ax.set_ylabel(t); ax.set_title(t + " by attrition")
save("06_boxplots")

cmp_cols = ["MonthlyIncome", "Age", "DistanceFromHome", "YearsAtCompany", "TotalWorkingYears", "YearsInCurrentRole",
            "YearsSinceLastPromotion", "YearsWithCurrManager", "JobLevel", "StockOptionLevel", "JobSatisfaction",
            "WorkLifeBalance", "EnvironmentSatisfaction", "JobInvolvement"]
comp = df.groupby("Attrition")[cmp_cols].median().T
comp.columns = ["Stayed (median)", "Left (median)"]
comp["Left vs Stayed %"] = ((comp["Left (median)"] / comp["Stayed (median)"] - 1) * 100).round(1)
comp["MannWhitney p"] = [stats.mannwhitneyu(df.loc[df.Attrition == "Yes", c], df.loc[df.Attrition == "No", c]).pvalue for c in cmp_cols]
print(comp.round(3))

**What it shows** (medians, tested with Mann-Whitney U because the data is skewed)
- **Income:** leavers have a median income of $3,202 versus $5,204 for stayers (-38%, p = 3.0e-14).
- **Age:** leavers are younger (median 32 vs 36).
- **Distance from home:** leavers live slightly farther away (median 9 vs 7 miles, p = 0.002). The effect is small compared with overtime.
- **Tenure:** leavers have a median of 3 years at the company versus 6, so attrition is concentrated in early-career, short-tenure staff.

### 5.5 Job satisfaction and work-life balance (Q5, Q9)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
order_s = ["Low", "Medium", "High", "Very High"]; order_w = ["Bad", "Good", "Better", "Best"]
sns.countplot(data=df, x="JobSatisfaction_lbl", hue="Attrition", order=order_s, palette=PAL, hue_order=["No", "Yes"], ax=axes[0])
axes[0].set_title("Job satisfaction vs attrition (counts)"); axes[0].set_xlabel("Job satisfaction")
sns.countplot(data=df, x="WorkLifeBalance_lbl", hue="Attrition", order=order_w, palette=PAL, hue_order=["No", "Yes"], ax=axes[1])
axes[1].set_title("Work-life balance vs attrition (counts)"); axes[1].set_xlabel("Work-life balance")
save("07_countplots_satisfaction_wlb")

js = df.groupby("JobSatisfaction_lbl", observed=True).Attrition_flag.mean().reindex(order_s) * 100
wl = df.groupby("WorkLifeBalance_lbl", observed=True).Attrition_flag.mean().reindex(order_w) * 100
print("Attrition rate (%) by job satisfaction:\n", js.round(1).to_string())
print("\nAttrition rate (%) by work-life balance:\n", wl.round(1).to_string())

**What it shows:** raw counts are dominated by group size, so the rates matter. Attrition falls from **22.8%** (low satisfaction) to **11.3%** (very high satisfaction). For work-life balance, employees rating it *Bad* leave at **31.2%** compared with **14.2%** for *Better*. Low satisfaction clearly carries the highest risk. Work-life balance is not a smooth ladder (*Best* is slightly higher than *Better*), but *Bad* stands out. The "Bad" group is small (only 80 employees), so treat its exact rate with caution.

### 5.6 Relationships between variables (bivariate / multivariate)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="Age", y="MonthlyIncome", hue="Attrition", palette=PAL, hue_order=["No", "Yes"], alpha=.6, ax=axes[0])
axes[0].set_title("Age vs monthly income (colour = attrition)")
sns.scatterplot(data=df, x="YearsAtCompany", y="MonthlyIncome", hue="Attrition", palette=PAL, hue_order=["No", "Yes"], alpha=.6, ax=axes[1])
axes[1].set_title("Years at company vs monthly income (colour = attrition)")
save("08_scatterplots")
print("Corr(Age, MonthlyIncome)            =", round(df.Age.corr(df.MonthlyIncome), 2))
print("Corr(YearsAtCompany, MonthlyIncome) =", round(df.YearsAtCompany.corr(df.MonthlyIncome), 2))
print("Corr(JobLevel, MonthlyIncome)       =", round(df.JobLevel.corr(df.MonthlyIncome), 2))

**What it shows:** income rises with age and tenure, and especially with **job level** (correlation with JobLevel = 0.95). The orange leavers cluster in the **bottom-left** of both plots: young, short-tenure, low-income employees. Very few high earners with long tenure leave.

In [ ]:
# Correlation heatmap (numeric variables + attrition flag)
num = df.select_dtypes("number")
corr = num.corr()
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".1f", annot_kws={"size": 7}, linewidths=.3, ax=ax)
ax.set_title("Correlation heatmap of numeric variables")
save("09_correlation_heatmap")
att_corr = corr["Attrition_flag"].drop("Attrition_flag").sort_values()
print("Strongest negative correlations with attrition:\n", att_corr.head(6).round(3).to_string())
print("\nStrongest positive correlations with attrition:\n", att_corr.tail(4).round(3).to_string())
print("\nHighly correlated predictor pairs (|r| > 0.7):")
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
print(pairs[pairs.abs() > .7].round(2).to_string())

**What it shows:** every single correlation with attrition is weak (the strongest is 0.17), so no single numeric variable predicts attrition on its own. The most negative links are with TotalWorkingYears, JobLevel and YearsInCurrentRole (r = -0.17, -0.17, -0.16): staff with higher level, income and experience tend to stay. The heatmap also shows **redundant variables** (e.g. JobLevel, MonthlyIncome and TotalWorkingYears move together), which would need handling before any predictive model.

### 5.7 Additional visualisations: age bands and combined risk factors

In [ ]:
# Additional chart 1: attrition rate by age group
ag = df.groupby("AgeGroup", observed=True).Attrition_flag.agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=ag.index.astype(str), y=ag["mean"] * 100, color="#4C78A8", ax=ax)
ax.axhline(overall, color="red", ls="--", label=f"company average {overall:.1f}%")
for i, (m, n) in enumerate(zip(ag["mean"] * 100, ag["count"])): ax.text(i, m + .4, f"{m:.1f}%\n(n={n})", ha="center", fontsize=9)
ax.set_xlabel("Age group"); ax.set_ylabel("Attrition rate (%)"); ax.set_ylim(0, (ag["mean"] * 100).max() + 6)
ax.set_title("Attrition rate by age group"); ax.legend()
save("10_attrition_by_age_group")

# Additional chart 2: combined risk heatmap (overtime x marital status)
piv = df.pivot_table(index="MaritalStatus", columns="OverTime", values="Attrition_flag", aggfunc="mean") * 100
cnt = df.pivot_table(index="MaritalStatus", columns="OverTime", values="Attrition_flag", aggfunc="count")
fig, ax = plt.subplots(figsize=(5.5, 4))
sns.heatmap(piv, annot=True, fmt=".1f", cmap="Reds", cbar_kws={"label": "Attrition rate (%)"}, ax=ax)
ax.set_title("Attrition rate (%): marital status x overtime")
save("11_heatmap_overtime_marital")
print(piv.round(1)); print(cnt); print(ag.round(3)); print("WLB bad n:", int((df.WorkLifeBalance==1).sum()))

In [ ]:
# Segment view: a "high-risk profile" combining the main risk factors found above
df["HighRisk"] = (df.OverTime == "Yes") & (df.MonthlyIncome < df.MonthlyIncome.median()) & (df.Age < 36)
seg = df.groupby("HighRisk").Attrition_flag.agg(["mean", "sum", "count"]).assign(rate=lambda d: d["mean"] * 100)
print(seg[["sum", "count", "rate"]].round(1))
single = df.groupby("MaritalStatus").Attrition_flag.mean() * 100
travel = df.groupby("BusinessTravel").Attrition_flag.mean() * 100
stock = df.groupby("StockOptionLevel").Attrition_flag.mean() * 100
print("\nBy marital status:\n", single.round(1).to_string(), "\n\nBy business travel:\n", travel.round(1).to_string(), "\n\nBy stock option level:\n", stock.round(1).to_string())

**What it shows**
- **Age:** attrition is highest for the **18-25** group (35.8%) and drops to its lowest at **36-45** (9.2%). It rises again for 46-55 (11.5%) and 56+ (17.0%, only 47 people, likely retirement-related), so the clearest flight risk is younger employees.
- **Overtime compounds other factors:** single employees who work overtime leave at **49.6%**, whereas married/divorced employees without overtime leave at only 8.4% / 6.1%.
- **High-risk profile** (overtime + below-median income + under 36): **51.5%** attrition (69 of 134), versus 12.6% for everyone else, a group that is only 9% of staff but accounts for 29% of all leavers.
- Also notable: stock-option level 0 has 24.4% attrition vs 9.4% for level 1, and frequent business travellers leave at 24.9% vs 8.0% for non-travellers.

### 5.8 Time-series analysis
**Not applicable.** This dataset is a single snapshot with no hire, exit or date columns, so there is no time dimension to plot. The assignment asks for a time-series chart "where applicable", and it isn't here. The closest proxy is tenure (`YearsAtCompany`), analysed above.

## 6. Insights & recommendations
### Key findings
1. **Overall attrition is 16.1%** (237 of 1470 employees).
2. **Overtime is the biggest driver:** 30.5% vs 10.4% attrition (2.9x).
3. **Younger, lower-paid, short-tenure employees leave most:** leavers earn about 38% less (median), are 4 years younger, and have 3 fewer years at the company.
4. **Role matters:** Sales Representative (39.8%) and Laboratory Technician (23.9%) are the hot spots; Research Director is the most stable (2.5%).
5. **Satisfaction and work-life balance** matter at the extremes: low job satisfaction = 22.8% attrition vs 11.3% for very high, and bad work-life balance = 31.2% (small group).
6. **Distance from home is a minor factor** compared with overtime and pay.
7. **A small high-risk group** (overtime + below-median income + under 36) is 9% of staff but 29% of leavers.

### Recommendations
1. **Control overtime.** Cap hours, hire or rebalance workload in roles with heavy overtime, and track overtime as an early-warning indicator.
2. **Target the high-risk segment:** young, early-career, below-median-income staff. Use stay interviews, mentoring and a visible career path in the first 2-3 years.
3. **Review pay for junior employees** in the highest-attrition roles and compare against market rates.
4. **Act on low satisfaction scores:** treat a low job-satisfaction or work-life-balance survey response as a trigger for a manager conversation.
5. **Consider stock-option or retention incentives** for level 0 employees, and review travel load for frequent travellers.
6. **Prioritise roles by cost of turnover**, not only by rate (see the counts in the job-role table).

### Limitations
- This is a **fictional, synthetic dataset** created by IBM, so the exact percentages should not be assumed to hold for a real company.
- The data is a **single snapshot**: no dates, so no trend or time-to-exit analysis.
- **Correlation is not causation.** For example, overtime may be a symptom of a deeper problem rather than the cause of leaving.
- **Class imbalance** (about 16% leavers) and small groups (e.g. work-life balance = Bad, n = 80 employees) make some rates unstable.
- No information on *why* people left (voluntary vs involuntary), and no external factors such as the job market.
- EDA shows association only. A predictive model (e.g. logistic regression) would be the natural next step.